In [4]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

print('Loading all three datasets...')

# Wildfire
wf = pd.read_csv('../data/wildfires/wildfires_processed.csv')
wf = wf[['LATITUDE', 'LONGITUDE', 'FIRE_YEAR', 'STAT_CAUSE_CODE', 
          'DISCOVERY_DOY', 'FIRE_SIZE_CLASS']].dropna()
wf['disaster_type'] = 0  # wildfire = 0
wf = wf.rename(columns={
    'LATITUDE': 'lat', 'LONGITUDE': 'lon',
    'FIRE_YEAR': 'year', 'STAT_CAUSE_CODE': 'feature3',
    'DISCOVERY_DOY': 'feature4', 'FIRE_SIZE_CLASS': 'severity'
})
# Encode wildfire severity A-G to 0-6
le = LabelEncoder()
wf['severity'] = le.fit_transform(wf['severity'])

# Flood
fl = pd.read_csv('../data/floods/floods_processed.csv')
fl = fl[['BEGIN_LAT', 'BEGIN_LON', 'EVENT_TYPE_CODE', 
          'STATE_CODE', 'severity']].dropna()
fl['disaster_type'] = 1  # flood = 1
fl['year'] = 2022
fl = fl.rename(columns={
    'BEGIN_LAT': 'lat', 'BEGIN_LON': 'lon',
    'EVENT_TYPE_CODE': 'feature3', 'STATE_CODE': 'feature4'
})

# Sample to balance
wf_sample = wf.sample(min(len(wf), 50000), random_state=42)
fl_sample = fl.sample(min(len(fl), 50000), random_state=42)

# Combine
features = ['lat', 'lon', 'year', 'feature3', 'feature4', 'disaster_type']
target = 'severity'

df_unified = pd.concat([
    wf_sample[features + [target]],
    fl_sample[features + [target]]
], ignore_index=True)

print(f'Wildfire samples: {len(wf_sample):,}')
print(f'Flood samples: {len(fl_sample):,}')
print(f'Combined shape: {df_unified.shape}')
print(f'Target distribution:')
print(df_unified['severity'].value_counts().sort_index())

# Add earthquake data to unified model
eq = pd.read_csv('../data/earthquakes/usgs_earthquakes.csv')
eq = eq[['latitude', 'longitude', 'depth', 'mag']].dropna()
eq['disaster_type'] = 2  # earthquake = 2
eq['year'] = 2022
eq['feature3'] = eq['depth']
eq['feature4'] = eq['mag']

# Create severity from magnitude
def eq_severity(mag):
    if mag >= 7.0: return 6
    elif mag >= 6.0: return 5
    elif mag >= 5.5: return 4
    elif mag >= 5.0: return 3
    elif mag >= 4.5: return 2
    else: return 1

eq['severity'] = eq['mag'].apply(eq_severity)
eq = eq.rename(columns={'latitude': 'lat', 'longitude': 'lon'})
eq_sample = eq.sample(min(len(eq), 50000), random_state=42)

# Combine all three
df_unified = pd.concat([
    wf_sample[features + [target]],
    fl_sample[features + [target]],
    eq_sample[features + [target]]
], ignore_index=True)

print(f'Earthquake samples: {len(eq_sample):,}')
print(f'Combined shape (all 3 disasters): {df_unified.shape}')

Loading all three datasets...
Wildfire samples: 50,000
Flood samples: 40,796
Combined shape: (90796, 7)
Target distribution:
severity
0    53535
1    29181
2     6685
3      725
4      360
5      225
6       85
Name: count, dtype: int64
Earthquake samples: 8,678
Combined shape (all 3 disasters): (99474, 7)


In [6]:
# Train unified model and compare against specialized models
X = df_unified[features]
y = df_unified[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

results = {}

# Unified Random Forest
print('\nTraining Unified Random Forest...')
start = time.time()
rf_unified = RandomForestClassifier(
    n_estimators=100, random_state=42,
    n_jobs=-1, class_weight='balanced'
)
rf_unified.fit(X_train, y_train)
rf_time = time.time() - start
rf_f1 = f1_score(y_test, rf_unified.predict(X_test), average='weighted')
results['Unified Random Forest'] = {'f1': rf_f1, 'time': rf_time}
print(f'  F1: {rf_f1:.4f} | Time: {rf_time:.1f}s')

# Unified XGBoost
print('Training Unified XGBoost...')
start = time.time()
xgb_unified = XGBClassifier(
    n_estimators=100, random_state=42,
    n_jobs=-1, verbosity=0
)
xgb_unified.fit(X_train, y_train)
xgb_time = time.time() - start
xgb_f1 = f1_score(y_test, xgb_unified.predict(X_test), average='weighted')
results['Unified XGBoost'] = {'f1': xgb_f1, 'time': xgb_time}
print(f'  F1: {xgb_f1:.4f} | Time: {xgb_time:.1f}s')

# Fair comparison — wildfire and flood only (both use F1)
# Earthquake uses RMSE which is a different metric — excluded from average
specialized_f1 = {
    'Wildfire RF (specialized)': 0.5881,
    'Flood RF (specialized)':    0.8445,
}

print('\n' + '='*58)
print(f'{"MODEL":<30} {"F1 SCORE":>10} {"TYPE":>14}')
print('='*58)

for model, r in results.items():
    print(f'{model:<30} {r["f1"]:>10.4f} {"unified":>14}')

for model, f1 in specialized_f1.items():
    print(f'{model:<30} {f1:>10.4f} {"specialized":>14}')

print(f'{"Earthquake TFT (specialized)":<30} {"RMSE=0.1176":>10} {"specialized":>14}')
print('='*58)

avg_specialized = sum(specialized_f1.values()) / len(specialized_f1)
avg_unified = sum(r["f1"] for r in results.values()) / len(results)

print(f'\nResearch Question: Does specialization beat generalization?')
print(f'Average specialized F1 (wildfire + flood): {avg_specialized:.4f}')
print(f'Average unified F1 (all 3 disasters):      {avg_unified:.4f}')

if avg_specialized > avg_unified:
    print('Answer: Specialized models outperform the unified model')
else:
    print('Answer: Unified model matches or beats specialized models')

print(f'\nNote: Earthquake TFT uses RMSE (0.1176), not F1 — it is a')
print(f'time-series regression task, not classification. Excluded from')
print(f'F1 average to ensure fair comparison.')
print(f'\nKey finding: Adding earthquake data to the unified model improved')
print(f'F1 from 0.7036 to {avg_unified:.4f} — diverse disaster training helps generalization.')

Training set: (79579, 6)
Test set: (19895, 6)

Training Unified Random Forest...
  F1: 0.7341 | Time: 1.4s
Training Unified XGBoost...
  F1: 0.7395 | Time: 0.9s

MODEL                            F1 SCORE           TYPE
Unified Random Forest              0.7341        unified
Unified XGBoost                    0.7395        unified
Wildfire RF (specialized)          0.5881    specialized
Flood RF (specialized)             0.8445    specialized
Earthquake TFT (specialized)   RMSE=0.1176    specialized

Research Question: Does specialization beat generalization?
Average specialized F1 (wildfire + flood): 0.7163
Average unified F1 (all 3 disasters):      0.7368
Answer: Unified model matches or beats specialized models

Note: Earthquake TFT uses RMSE (0.1176), not F1 — it is a
time-series regression task, not classification. Excluded from
F1 average to ensure fair comparison.

Key finding: Adding earthquake data to the unified model improved
F1 from 0.7036 to 0.7368 — diverse disaster train